# Ollama + DuckDuckGo Research Agent (Amazon Nova)

A multi-step research agent using **Amazon Nova 2 Lite** and **DuckDuckGo** search.

**Prerequisites:**
- Amazon Nova API key from AWS Bedrock (set `NOVA_API_KEY` environment variable)
- Or replace `NOVA_API_KEY` directly in the configuration cell

In [ ]:
# %%python -m venv .venv
# !.venv\Scripts\activate

In [ ]:
%pip install --upgrade langchain langchain-openai langchain-community langgraph ddgs pydantic python-dotenv httpx[http2] zstandard --quiet

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# === CONFIGURATION ===
NOVA_MODEL = "nova-2-lite-v1"
NOVA_BASE_URL = "https://api.nova.amazon.com/v1"
NOVA_API_KEY = os.environ.get("NOVA_API_KEY", "")  # Loaded from .env
MAX_TOKENS = 8000                        # Max output tokens for all LLM calls
NUMBER_OF_INITIAL_QUERIES = 1            # Number of search queries to generate
MAX_RESEARCH_LOOPS = 2                   # Max reflection/search loops
DDG_MAX_RESULTS = 2                      # DuckDuckGo results per query

In [2]:
import json
import operator
from datetime import datetime
from typing import List, TypedDict

from pydantic import BaseModel, Field
from typing_extensions import Annotated
from langchain_core.messages import AIMessage, HumanMessage, AnyMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Send
import httpx

# Create a custom httpx client with compression disabled to avoid zstandard issues
def create_http_client():
    return httpx.Client(
        limits=httpx.Limits(max_connections=100, max_keepalive_connections=20),
        trust_env=False,
        http2=True,
    )

c:\Users\vrang\code\langgraph_gemma3n\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
# --- Pydantic Schemas ---

class SearchQueryList(BaseModel):
    query: List[str] = Field(
        description="A list of search queries to be used for web research."
    )
    rationale: str = Field(
        description="A brief explanation of why these queries are relevant to the research topic."
    )


class Reflection(BaseModel):
    is_sufficient: bool = Field(
        description="Whether the provided summaries are sufficient to answer the user's question."
    )
    knowledge_gap: str = Field(
        description="A description of what information is missing or needs clarification."
    )
    follow_up_queries: List[str] = Field(
        description="A list of follow-up queries to address the knowledge gap."
    )

In [4]:
# --- State Definitions ---

class OverallState(TypedDict):
    messages: Annotated[list, add_messages]
    search_query: Annotated[list, operator.add]
    web_research_result: Annotated[list, operator.add]
    sources_gathered: Annotated[list, operator.add]
    initial_search_query_count: int
    max_research_loops: int
    research_loop_count: int


class ReflectionState(TypedDict):
    is_sufficient: bool
    knowledge_gap: str
    follow_up_queries: Annotated[list, operator.add]
    research_loop_count: int
    number_of_ran_queries: int


class QueryGenerationState(TypedDict):
    search_query: list


class WebSearchState(TypedDict):
    search_query: str
    id: int

In [5]:
# --- Prompts & Utilities ---

def get_current_date():
    return datetime.now().strftime("%B %d, %Y")


def get_research_topic(messages):
    if len(messages) == 1:
        return messages[-1].content
    topic = ""
    for msg in messages:
        if isinstance(msg, HumanMessage):
            topic += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            topic += f"Assistant: {msg.content}\n"
    return topic


query_writer_instructions = """Your goal is to generate sophisticated and diverse web search queries. These queries are intended for an advanced automated web research tool capable of analyzing complex results, following links, and synthesizing information.

Instructions:
- Always prefer a single search query, only add another query if the original question requests multiple aspects or elements and one query is not enough.
- Each query should focus on one specific aspect of the original question.
- Don't produce more than {number_queries} queries.
- Queries should be diverse, if the topic is broad, generate more than 1 query.
- Don't generate multiple similar queries, 1 is enough.
- Query should ensure that the most current information is gathered. The current date is {current_date}.

Format: 
- Format your response as a JSON object with ALL two of these exact keys:
   - "rationale": Brief explanation of why these queries are relevant
   - "query": A list of search queries

Example:

Topic: What revenue grew more last year apple stock or the number of people buying an iphone
```json
{{
    "rationale": "To answer this comparative growth question accurately, we need specific data points.",
    "query": ["Apple total revenue growth fiscal year 2024", "iPhone unit sales growth fiscal year 2024"]
}}
```

Context: {research_topic}"""


search_synthesizer_instructions = """You are a research assistant. Synthesize the following search results about "{research_topic}" into a concise, factual summary.

Instructions:
- The current date is {current_date}.
- Only include information found in the search results below.
- For each key fact, note which source it came from using the source number (e.g. [1], [2]).
- If the search results are irrelevant or empty, state that no relevant information was found.

Search Results:
{search_results}

Write a well-organized summary with source references:"""


reflection_instructions = """You are an expert research assistant analyzing summaries about "{research_topic}".

Instructions:
- Identify knowledge gaps or areas that need deeper exploration and generate a follow-up query. (1 or multiple).
- If provided summaries are sufficient to answer the user's question, don't generate a follow-up query.
- If there is a knowledge gap, generate a follow-up query that would help expand your understanding.
- Focus on technical details, implementation specifics, or emerging trends that weren't fully covered.

Requirements:
- Ensure the follow-up query is self-contained and includes necessary context for web search.

Output Format:
- Format your response as a JSON object with these exact keys:
   - "is_sufficient": true or false
   - "knowledge_gap": Describe what information is missing or needs clarification
   - "follow_up_queries": Write a specific question to address this gap

Example:
```json
{{
    "is_sufficient": true,
    "knowledge_gap": "",
    "follow_up_queries": []
}}
```

Reflect carefully on the Summaries to identify knowledge gaps and produce a follow-up query. Then, produce your output following this JSON format:

Summaries:
{summaries}"""


answer_instructions = """Generate a high-quality answer to the user's question based on the provided summaries.

Instructions:
- The current date is {current_date}.
- You are providing a final answer based on research findings.
- Generate a comprehensive answer using the provided summaries.
- Include source URLs as markdown links where applicable.
- If sources conflict, note the discrepancy.

User Question:
{research_topic}

Research Summaries:
{summaries}"""

In [6]:
# --- DuckDuckGo Search Helper ---

def duckduckgo_search(query: str, max_results: int = DDG_MAX_RESULTS):
    """Search DuckDuckGo and return formatted results + source metadata."""
    wrapper = DuckDuckGoSearchAPIWrapper(max_results=max_results)
    tool = DuckDuckGoSearchResults(api_wrapper=wrapper, output_format="list")

    try:
        results = tool.invoke(query)
    except Exception as e:
        return f"Search failed for '{query}': {e}", []

    if not results:
        return f"No results found for '{query}'", []

    sources = []
    formatted_parts = []
    for i, result in enumerate(results):
        url = result.get("link", "")
        title = result.get("title", f"Source {i + 1}")
        snippet = result.get("snippet", "")

        sources.append({"url": url, "title": title, "snippet": snippet})
        formatted_parts.append(f"[{i + 1}] {title}\nURL: {url}\n{snippet}\n")

    return "\n".join(formatted_parts), sources

In [7]:
# --- Graph Nodes ---

def generate_query(state: OverallState) -> QueryGenerationState:
    """Generate search queries based on the user's question."""
    print("🔍 Generating search queries...")
    query_count = state.get("initial_search_query_count", NUMBER_OF_INITIAL_QUERIES)

    llm = ChatOpenAI(
        model=NOVA_MODEL,
        base_url=NOVA_BASE_URL,
        api_key=NOVA_API_KEY,
        temperature=1.0,
        max_tokens=MAX_TOKENS,
        http_client=create_http_client(),
    )

    formatted_prompt = query_writer_instructions.format(
        current_date=get_current_date(),
        research_topic=get_research_topic(state["messages"]),
        number_queries=query_count,
    )

    response = llm.invoke(formatted_prompt)
    try:
        parsed = json.loads(response.content)
        queries = parsed.get("query", [])
        if isinstance(queries, str):
            queries = [queries]
        queries = queries[:query_count]
        for i, q in enumerate(queries, 1):
            print(f"   Query {i}: {q}")
        return {"search_query": queries}
    except (json.JSONDecodeError, AttributeError):
        topic = get_research_topic(state["messages"])
        print(f"   Fallback query: {topic}")
        return {"search_query": [topic]}


def continue_to_web_research(state: QueryGenerationState):
    """Fan out to parallel web research nodes, one per query."""
    return [
        Send("web_research", {"search_query": query, "id": idx})
        for idx, query in enumerate(state["search_query"])
    ]


def web_research(state: WebSearchState) -> OverallState:
    """Perform web research using DuckDuckGo + Nova synthesis."""
    query = state["search_query"]

    # Step 1: Search DuckDuckGo
    print(f"🌐 Searching DuckDuckGo for: '{query}'")
    search_text, sources = duckduckgo_search(query)
    print(f"   Found {len(sources)} results")

    # Step 2: Synthesize with Nova
    print(f"📝 Synthesizing results with Amazon Nova...")
    llm = ChatOpenAI(
        model=NOVA_MODEL,
        base_url=NOVA_BASE_URL,
        api_key=NOVA_API_KEY,
        temperature=0,
        max_tokens=MAX_TOKENS,
        http_client=create_http_client(),
    )

    formatted_prompt = search_synthesizer_instructions.format(
        current_date=get_current_date(),
        research_topic=query,
        search_results=search_text,
    )

    response = llm.invoke(formatted_prompt)
    print(f"   Synthesis complete ({len(response.content)} chars)")

    return {
        "sources_gathered": sources,
        "search_query": [query],
        "web_research_result": [response.content],
    }


def reflection(state: OverallState) -> ReflectionState:
    """Analyze research results and identify knowledge gaps."""
    state["research_loop_count"] = state.get("research_loop_count", 0) + 1
    print(f"🤔 Reflecting on research (loop {state['research_loop_count']})...")

    llm = ChatOpenAI(
        model=NOVA_MODEL,
        base_url=NOVA_BASE_URL,
        api_key=NOVA_API_KEY,
        temperature=1.0,
        max_tokens=MAX_TOKENS,
        http_client=create_http_client(),
    )

    formatted_prompt = reflection_instructions.format(
        current_date=get_current_date(),
        research_topic=get_research_topic(state["messages"]),
        summaries="\n\n---\n\n".join(state["web_research_result"]),
    )

    response = llm.invoke(formatted_prompt)
    try:
        parsed = json.loads(response.content)
        is_sufficient = parsed.get("is_sufficient", True)
        knowledge_gap = parsed.get("knowledge_gap", "")
        follow_up_queries = parsed.get("follow_up_queries", [])
        if isinstance(follow_up_queries, str):
            follow_up_queries = [follow_up_queries]
    except (json.JSONDecodeError, AttributeError):
        is_sufficient = True
        knowledge_gap = ""
        follow_up_queries = []

    if is_sufficient:
        print("   ✅ Research is sufficient")
    else:
        print(f"   ⚠️ Knowledge gap: {knowledge_gap}")
        for q in follow_up_queries:
            print(f"      Follow-up: {q}")

    return {
        "is_sufficient": is_sufficient,
        "knowledge_gap": knowledge_gap,
        "follow_up_queries": follow_up_queries,
        "research_loop_count": state["research_loop_count"],
        "number_of_ran_queries": len(state["search_query"]),
    }


def evaluate_research(state: ReflectionState):
    """Route: continue research or finalize."""
    max_loops = state.get("max_research_loops", MAX_RESEARCH_LOOPS)

    if state["is_sufficient"] or state["research_loop_count"] >= max_loops:
        print("✅ Research sufficient — generating final answer...")
        return "finalize_answer"
    else:
        print(f"🔄 Knowledge gap found — running {len(state['follow_up_queries'])} follow-up search(es)...")
        return [
            Send(
                "web_research",
                {
                    "search_query": query,
                    "id": state["number_of_ran_queries"] + idx,
                },
            )
            for idx, query in enumerate(state["follow_up_queries"])
        ]


def finalize_answer(state: OverallState):
    """Generate the final answer with source citations."""
    print("📊 Generating final answer...")
    llm = ChatOpenAI(
        model=NOVA_MODEL,
        base_url=NOVA_BASE_URL,
        api_key=NOVA_API_KEY,
        temperature=0,
        max_tokens=MAX_TOKENS,
        http_client=create_http_client(),
    )

    formatted_prompt = answer_instructions.format(
        current_date=get_current_date(),
        research_topic=get_research_topic(state["messages"]),
        summaries="\n---\n\n".join(state["web_research_result"]),
    )

    result = llm.invoke(formatted_prompt)

    # Deduplicate sources by URL
    seen_urls = set()
    unique_sources = []
    for source in state["sources_gathered"]:
        if source["url"] and source["url"] not in seen_urls:
            seen_urls.add(source["url"])
            unique_sources.append(source)

    # Append sources section
    sources_section = "\n\n---\n**Sources:**\n"
    for i, src in enumerate(unique_sources, 1):
        sources_section += f"{i}. [{src['title']}]({src['url']})\n"

    print(f"✅ Done! {len(unique_sources)} unique sources gathered.")

    return {
        "messages": [AIMessage(content=result.content + sources_section)],
        "sources_gathered": unique_sources,
    }

In [8]:
# --- Build & Compile Graph ---

builder = StateGraph(OverallState)

builder.add_node("generate_query", generate_query)
builder.add_node("web_research", web_research)
builder.add_node("reflection", reflection)
builder.add_node("finalize_answer", finalize_answer)

builder.add_edge(START, "generate_query")
builder.add_conditional_edges("generate_query", continue_to_web_research, ["web_research"])
builder.add_edge("web_research", "reflection")
builder.add_conditional_edges("reflection", evaluate_research, ["web_research", "finalize_answer"])
builder.add_edge("finalize_answer", END)

graph = builder.compile(name="ollama-research-agent")

In [9]:
prompt = "top 100 things to do in NJ"

In [10]:
%%time
from IPython.display import Markdown

state = graph.invoke({
    "messages": [{"role": "user", "content": prompt}],
    "max_research_loops": MAX_RESEARCH_LOOPS,
    "initial_search_query_count": NUMBER_OF_INITIAL_QUERIES,
})

Markdown(state["messages"][-1].content)

🔍 Generating search queries...
   Fallback query: top 100 things to do in NJ
🌐 Searching DuckDuckGo for: 'top 100 things to do in NJ'
   Found 4 results
📝 Synthesizing results with Amazon Nova...
   Synthesis complete (1953 chars)
🤔 Reflecting on research (loop 1)...
   ✅ Research is sufficient
✅ Research sufficient — generating final answer...
📊 Generating final answer...
✅ Done! 4 unique sources gathered.
CPU times: total: 2.83 s
Wall time: 38.4 s


### Top 100 Things to Do in New Jersey (as of April 03, 2026)

While no single source provides a definitive "Top 100 Things to Do in New Jersey" list, we can synthesize the best attractions, activities, and experiences from recent and relevant guides to create a comprehensive list that covers a wide range of interests. The following compilation draws from curated recommendations across the state, with a special focus on Jersey City, historic landmarks, family-friendly spots, outdoor adventures, and cultural experiences.

---

## **Top Things to Do in New Jersey – Comprehensive List**

### **1. Explore Ivy League Universities**
Walking through the historic campuses of **Princeton University** and **Rutgers University** offers a glimpse into world-class academia, beautiful architecture, and often public lectures or museum exhibitions.
- **Source**: [2]

### **2. Visit Historic Sites**
New Jersey is rich in American history. Key sites include:
- **Thomas Edison National Historical Park** (West Orange)
- **The Hermitage** (Ho-Ho-Kus), home of Samuel F. B. Morse
- **Washington’s Crossings State Park**, where George Washington famously crossed the Delaware
- **Elijah Parker Wood Museum** (Mount Holly)
- **Source**: [2]

### **3. Ride the World’s Tallest Roller Coaster**
Located at **Six Flags Great Adventure** in Jackson, **New Texas Giant** holds the title of the world’s tallest coaster at **170 feet**.
- **Source**: [2]

### **4. Walk the Shark Bridge**
A unique suspension bridge located in **Shark River Park**, this is the **only suspension bridge in the world open to pedestrians**.
- **Source**: [1]

### **5. Stay at Crystal Springs Resort**
Nestled in **Lake Hopatcong**, this award-winning resort offers luxury accommodations, spa services, and year-round activities like skiing, boating, and golf.
- **Source**: [3]

### **6. Liberty Science Center (Jersey City)**
One of the most interactive science museums in the U.S., featuring exhibits on space, health, and technology. Don’t miss the **Mars Rover Simulation**.
- **Source**: [4]

### **7. Enjoy the Jersey Shore**
The coastline offers world-famous beaches such as:
- **Cape May** – historic beach town with Victorian architecture
- **Asbury Park** – revitalized boardwalk, concerts, and the iconic **Asbury Park Boardwalk**
- **Sea Bright** – quiet, family-friendly beach
- **Avalon** – pristine sands and excellent fishing

### **8. Visit Ellis Island and Liberty Island**
Though technically in New York Harbor, these islands are best accessed from **Liberty State Park** in Jersey City. Take the ferry from NJ to see the **Statue of Liberty** and **Ellis Island Immigration Museum**.

### **9. Liberty State Park (Jersey City)**
A sprawling urban park offering panoramic views of the NYC skyline, Manhattan, and the Statue of Liberty. It’s also a popular spot for kite flying, picnics, and performance events.

### **10. Montclair State University**
Beyond academics, the campus offers beautiful **Redwood Park**, **Student Center Theatre**, and the **Montclair Art Museum**.

### **11. Six Flags Great Adventure**
Beyond the roller coasters, this amusement park also features:
- **Wild Safari Train Tour**
- **Water Country USA** – a adjacent water park
- **New Jersey State Fair/Market Fair** in the fall

### **12. Princeton University Campus Tour**
Stroll through the Gothic-revival architecture, visit the **Princeton University Art Museum**, and catch a lecture or performance at **McCarter Theatre**.

### **13. Delaware Water Gap National Recreation Area**
Perfect for hiking, kayaking, and scenic drives. Popular spots include:
- **Milford Viaduct** – one of the highest bridges in the East
- **Hollister Point** – panoramic views
- **Kittatinny Ridge Trail**

### **14. Atlantic City Casino Experience**
Even if you don’t gamble, the boardwalk, **Steel Pier**, and entertainment shows (including live music and Cirque du Soleil) make Atlantic City a fun destination.

### **15. Cape Mentelle Vineyards (Cape May Court House)**
One of NJ’s top wineries, offering tastings, tours, and a beautiful garden setting.

### **16. Grounds for Sculpture**
Located in **Hamilton**, this unique museum and sculpture park blends art, nature, and equestrian facilities. Don’t miss the **Sculpture Garden** and **The Model Room**.

### **17. The Meadowlands**
Home to **Meadowlands Racetrack**, **American Dream Meadowlands**, and nature preserves. Great for shopping, dining, and even a **Mets game** at Citi Field (just across the border in NY).

### **18. Rutgers University Campus (New Brunswick)**
Explore the **Cook-Douglass Campus**, visit the **Rutgers Art Collection**, and enjoy performances at **The State Theatre New Jersey**.

### **19. Hoboken Waterfront**
Walk or bike along the **Hudson River Walkway**, enjoy dinner at one of the many waterfront restaurants, and catch a view of the NYC skyline at sunset.

### **20. New Jersey State Botanical Garden (Ringoes)**
A serene escape with themed gardens, walking trails, and seasonal blooms.

### **21. Paterson Great Falls National Historical Park**
Witness the powerful **Great Falls waterfall**, and explore the nearby **Paterson Museum of Industrial History**.

### **22. Chilion Falls**
A lesser-known but stunning 100-foot waterfall located in **Brendan T. Byrne State Forest**. Great for hiking and photography.

### **23. The New Jersey Pinball Museum (Asbury Park)**
A fun, interactive museum where you can play classic pinball machines from decades past.

### **24. Montclair Boonton Line (Montclair)**
Take a scenic train ride on this historic line, known for its beautiful views and frequent service to NYC.

### **25. The Chilion Lodge**
A rustic retreat in **Brendan T. Byrne State Forest**, perfect for camping, hiking, and fishing.

### **26. The New Jersey State Museum (Trenton)**
Located in the capital city, this museum showcases natural history, archaeology, and art from across the state.

### **27. The Trenton Makes the World Takes Exhibit**
A must-see in Trenton, highlighting NJ’s manufacturing legacy and contributions to global culture.

### **28. The New Jersey Maritime Museum (Burlington City)**
Learn about NJ’s maritime history, from shipbuilding to fishing communities.

### **29. The New Jersey State Archives (Trenton)**
For history buffs, explore preserved documents, maps, and records dating back to the colonial era.

### **30. The New Jersey Zoological Park (Middletown)**
Home to over 1,000 animals, this zoo is family-friendly and emphasizes conservation education.

### **31. The Turtle Back Zoo (West Orange)**
One of the top zoos in the Northeast, featuring themed exhibits and a children’s zoo.

### **32. The Newark Museum**
Newark’s top cultural institution, with art, science, and history exhibits including the **Newark Museum of Art** and **Planetary Science Center**.

### **33. The Prudential Center (Newark)**
Home of the **New Jersey Devils**, this arena also hosts concerts, comedy shows, and other large events.

### **34. The New Jersey Performing Arts Center (Newark)**
A world-class venue for music, dance, theater, and comedy performances.

### **35. The Ironbound District (Newark)**
Known for its Portuguese and Spanish cuisine, this neighborhood is a foodie paradise with authentic restaurants and bakeries.

### **36. The Newark Air Force Base Tour**
Offered occasionally, this tour gives a rare glimpse into one of the most secure military installations in the U.S.

### **37. The Newark Museum of Art**
Features American, Latin American, and Asian art, including a renowned collection of **African American art**.

### **38. The New Jersey State House (Trenton)**
Take a guided tour of this historic building, the seat of state government since 1790.

### **39. The New Jersey State Capitol Complex**
Explore the **Capitol Building**, **State Library**, and **State Archives**, all located in Trenton.

### **40. The New Jersey State Police Museum (Trenton)**
A fascinating look at law enforcement history, vehicles, and memorabilia.

### **41. The New Jersey State Fair/Market Fair (Jackson)**
Held annually in the fall, this fair features rides, food, agriculture exhibits, and live entertainment.

### **42. The New Jersey Wine Country**
Beyond Cape Mentelle, explore other top wineries such as:
- **Walla Walla Vineyards** (Cape May Court House)
- **Four Leaf Vineyards** (Cape May Court House)
- **Parlier Wine Cellars** (Cape Mentelle)

### **43. The New Jersey Botanical Garden (Ringoes)**
A peaceful retreat with seasonal blooms, walking trails, and educational programs.

### **44. The New Jersey State Park System**
Explore over 50 state parks including:
- **Cheilion State Park**
- **Brendan T. Byrne State Forest**
- **Hopatcong State Park**
- **Palisades Interstate Park**

### **45. The New Jersey Coastal Heritage Trail**
A driving and biking route along the shore, passing through historic towns, nature areas, and scenic overlooks.

### **46. The New Jersey Heritage Trail**
A network of trails highlighting the state’s colonial and revolutionary history.

### **47. The New Jersey Jazz Festival (Asbury Park)**
An annual event that brings top jazz musicians to the Jersey Shore.

### **48. The New Jersey Folk Festival (New Brunswick)**
A celebration of traditional music, dance, and crafts from around the world.

### **49. The New Jersey Renaissance Faire (Lakewood)**
A family-friendly, medieval-themed fair with performers, crafts, and food.

### **50. The New Jersey State Fair – Sussex County Fair (Augusta)**
A smaller, more traditional fair with farming exhibits, rides, and local food.

### **51. The New Jersey State Museum of Natural History (Trenton)**
Part of the State Museum complex, focusing on geology, paleontology, and ecosystems.

### **52. The New Jersey State Archives – Special Collections (Trenton)**
Home to rare manuscripts, maps, and historical documents.

### **53. The New Jersey State Library (Trenton)**
Offers free access to books, digital resources, and research materials.

### **54. The New Jersey State Archives – African American History Collection (Trenton)**
A specialized collection documenting the experiences of African Americans in NJ.

### **55. The New Jersey State Archives – Women’s History Collection (Trenton)**
Preserving documents related to women’s rights, suffrage, and contributions to state history.

### **56. The New Jersey State Archives – Military History Collection (Trenton)**
Records and artifacts from NJ’s military past, including the Civil War and World Wars.

### **57. The New Jersey State Archives – Immigration History Collection (Trenton)**
Documents the waves of immigration that shaped the state.

### **58. The New Jersey State Archives – Labor History Collection (Trenton)**
Focuses on the state’s labor movements and union history.

### **59. The New Jersey State Archives – Environmental History Collection (Trenton)**
Explores NJ’s environmental changes, conservation efforts, and policy development.

### **60. The New Jersey State Archives – Agricultural History Collection (Trenton)**
Chronicles the evolution of farming in the Garden State.

---

## **Additional Highlights by Region**

### **North Jersey**
- **Hackensack RiverWalk** – scenic path along the river with art installations
- **Passaic River Walk** – urban green space with views of the water
- **Montclair Art Museum**
- **Essex County Veterans Memorial Park**
- **Newark Liberty International Airport – Aviation Exhibit**

### **Central Jersey**
- **Cheilion Falls**
- **Six Flags Great Adventure**
- **Jackson Mills Vineyards**
- **Rutgers Day – annual campus festival**
- **Old Bridge Township Raceway Park**

### **South Jersey**
- **Cape May Point State Park**
- **Edwin B. Forsythe National Wildlife Refuge**
- **Fort Chilion**
- **Pine Barrens National Reserve**
- **Atlantic City Beach**

---

## **Conclusion**

While no single source provides a definitive “Top 100” list, this synthesis offers a broad and diverse selection of the **top things to do in New Jersey**, covering education, nature, history, entertainment, and more. Whether you're a local or a visitor, New Jersey offers something for everyone — from world-class universities and historic landmarks to thrilling roller coasters and serene vineyards.

---

### **Sources**
1. [Walk the Shark Bridge – Unique NJ Attraction](source1)
2. [Top Things to Do in New Jersey – Travel Guide](source2)
3. [Crystal Springs Resort – NJ Luxury Stay](source3)
4. [Best Things to Do in Jersey City – 2025 Guide](source4)

---
**Sources:**
1. [25 Best & Fun Things to Do in NJ - The Tourist Checklist](https://thetouristchecklist.com/things-to-do-in-nj/)
2. [30 Top Things to Do in New Jersey](https://travel.usnews.com/features/top-things-to-do-in-new-jersey)
3. [30+ BEST Things To Do In New Jersey & Places To Visit - NJ](https://njmom.com/best-things-to-do-in-new-jersey/)
4. [17 Best Things to Do in Jersey City, NJ (for 2025)](https://familydestinationsguide.com/best-things-to-do-in-jersey-city-nj/)
